In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 23. KV Cache — The Core of Inference Speed [CPU/GPU Benchmark ⑩]

> **Learning Goals**
> - Understand why KV Cache reduces inference cost from $O(n^2)$ to $O(n)$
> - Distinguish Prefill and Decode phases
> - Compute KV Cache memory usage
> - Compare generation speed with and without KV Cache

## 23.1 Inference Phases: Prefill vs Decode

LLM inference has two phases:
1. **Prefill (Context Encoding)**: process the whole prompt. Parallelizable. Cost $O(n^2)$.
2. **Decode (Token Generation)**: generate tokens one by one. Sequential. Cost $O(n)$ per step.

Decode dominates total latency. KV Cache greatly improves decode speed.

## 23.2 Why Cache Only K and V?

Attention: $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$

When generating token $t$:
- Need $Q_t$ (new query)
- Need $K_{\leq t}, V_{\leq t}$ (all previous keys and values)

Without cache, every step recomputes all previous $K,V$ values, leading to $O(t^2)$ work. With cache, store $K_{\leq t-1}, V_{\leq t-1}$ and append only $K_t,V_t$, giving $O(t)$ work.

**Q is not cached** because each step only needs the new $Q_t$.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

# Attention without cache: recompute the full context at every step
def attention_no_cache(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# attention with cache (incremental)
def attention_with_cache(q_new, k_cache, v_cache, k_new, v_new):
    """q_new: (B, 1, d), k_cache/v_cache: (B, T, d)"""
    # append new k and v to the cache
    k_full = torch.cat([k_cache, k_new], dim=1)  # (B, T+1, d)
    v_full = torch.cat([v_cache, v_new], dim=1)
    d_k = q_new.shape[-1]
    scores = q_new @ k_full.transpose(-1, -2) / (d_k ** 0.5)  # (B, 1, T+1)
    weights = F.softmax(scores, dim=-1)
    out = weights @ v_full  # (B, 1, d)
    return out, k_full, v_full

# Comparison: 100 tokens Generation
np.random.seed(0)
torch.manual_seed(0)
B, d, T = 1, 64, 1
n_generate = 50

# without cache
t0 = time.perf_counter()
Q = torch.randn(B, T, d)
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)
for step in range(n_generate):
    # recompute the whole context at each step
    Q_new = torch.randn(B, 1, d)
    Q = torch.cat([Q, Q_new], dim=1)
    K = torch.cat([K, torch.randn(B, 1, d)], dim=1)
    V = torch.cat([V, torch.randn(B, 1, d)], dim=1)
    out = attention_no_cache(Q, K, V)
t_no_cache = time.perf_counter() - t0
print(f"cache disabled: {t_no_cache*1000:.2f}ms ({n_generate} tokens)")

# with cache
torch.manual_seed(0)
t0 = time.perf_counter()
k_cache = torch.randn(B, 1, d)
v_cache = torch.randn(B, 1, d)
for step in range(n_generate):
    q_new = torch.randn(B, 1, d)
    k_new = torch.randn(B, 1, d)
    v_new = torch.randn(B, 1, d)
    out, k_cache, v_cache = attention_with_cache(q_new, k_cache, v_cache, k_new, v_new)
t_cache = time.perf_counter() - t0
print(f"cache enabled: {t_cache*1000:.2f}ms ({n_generate} tokens)")
print(f"Speed Improvement: {t_no_cache/t_cache:.2f}x")


## 23.3 KV Cache Memory Calculation

KV Cache memory = $2 \cdot L \cdot h \cdot d_k \cdot n \cdot \text{dtype}$

- $L$: num of layers
- $h$: num of KV heads (MHA: $h$, MQA: 1, GQA: $g$)
- $d_k$: head dimension
- $n$: sequence length
- 2: K and V
- dtype: FP16=2 bytes, FP32=4 bytes

Example: LLaMA-7B ($L=32, h=32, d_k=128$), FP16, $n=4096$:
$$2 \times 32 \times 32 \times 128 \times 4096 \times 2 = 2 \text{ GB}$$

For $n=32768$, this becomes 16GB and can occupy most GPU memory.


In [ ]:
# KV Cache memory calculator
def kv_cache_memory_gb(n_layers, n_kv_heads, d_k, seq_len, dtype_bytes=2):
    """KV Cache Memory (GB)."""
    return 2 * n_layers * n_kv_heads * d_k * seq_len * dtype_bytes / (1024**3)

# various models and sequence lengths
models = [
    ('LLaMA-7B', 32, 32, 128, 2),
    ('LLaMA-13B', 40, 40, 128, 2),
    ('LLaMA-70B', 80, 8, 128, 2),  # GQA
    ('GPT-2 small', 12, 12, 64, 2),
    ('Mistral-7B', 32, 8, 128, 2),  # GQA
]
seq_lens = [2048, 8192, 32768, 131072]

print(f"{'Model':>15} | ", end='')
for sl in seq_lens:
    print(f"n={sl:>6}", end=' | ')
print()
print("-" * 75)
for name, L, h, dk, dt in models:
    print(f"{name:>15} | ", end='')
    for sl in seq_lens:
        mem = kv_cache_memory_gb(L, h, dk, sl, dt)
        print(f"{mem:>7.2f}GB", end=' | ')
    print()

print("\n=> As sequence length grows, KV Cache occupies a large part of memory.")
print("=> GQA(Mistral, LLaMA-70B) fewer KV heads make memory use more efficient.")


## 23.4 [CPU/GPU Benchmark ⑩] KV Cache On/Off


In [ ]:
# KV Cache benchmark
from llm_math.bench import time_fn

# measure cache effect in a single attention layer
def bench_decode_no_cache(n_context, d=64, n_decode=20):
    """without cache n_decode tokens Generation."""
    Q = torch.randn(1, n_context, d)
    K = torch.randn(1, n_context, d)
    V = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        Q_new = torch.randn(1, 1, d)
        Q = torch.cat([Q, Q_new], dim=1)
        K = torch.cat([K, torch.randn(1, 1, d)], dim=1)
        V = torch.cat([V, torch.randn(1, 1, d)], dim=1)
        _ = Q @ K.transpose(-1, -2) / (d ** 0.5) @ V

def bench_decode_with_cache(n_context, d=64, n_decode=20):
    """with cache n_decode tokens Generation."""
    k_cache = torch.randn(1, n_context, d)
    v_cache = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        q_new = torch.randn(1, 1, d)
        k_new = torch.randn(1, 1, d)
        v_new = torch.randn(1, 1, d)
        k_full = torch.cat([k_cache, k_new], dim=1)
        v_full = torch.cat([v_cache, v_new], dim=1)
        _ = q_new @ k_full.transpose(-1, -2) / (d ** 0.5) @ v_full
        k_cache = k_full
        v_cache = v_full

print(f"{'n_context':>10} | {'No Cache (ms)':>15} | {'Cache (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n_ctx in [128, 512, 1024, 2048]:
    t_no = time_fn(bench_decode_no_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    t_yes = time_fn(bench_decode_with_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    print(f"{n_ctx:>10} | {t_no:>15.3f} | {t_yes:>12.3f} | {t_no/t_yes:>9.2f}x")
print("\n=> KV Cache becomes more effective as context length grows (O(n) vs O(n^2)).")


## 23.5 Continuous Batching

Classic batching requires all sequences in a batch to finish together, so short requests wait for long ones.

**Continuous Batching** (vLLM, etc.) immediately replaces finished sequences with new requests.
- Greatly improves throughput
- Handles variable-length requests

## 23.6 PagedAttention (vLLM)

Inspired by OS paging:
- Split KV Cache into fixed-size pages
- Store pages in non-contiguous memory
- Manage them with a page table

This removes memory fragmentation and enables larger batches. It is a major reason vLLM is 2-4x faster.

## 23.7 Key Takeaways

| Concept | Meaning |
|---|---|
| Prefill | parallel prompt processing, $O(n^2)$ |
| Decode | token generation, $O(n)$ per token with KV cache |
| KV Cache | stores $K,V$ and appends only $K_t,V_t$ each step |
| Memory | $2 L h d_k n \cdot \text{dtype}$ |
| Continuous Batching | variable lengths, high throughput |
| PagedAttention | solves memory fragmentation |

## Exercises

1. Compare generation time for 100 tokens at n=128, 512, and 2048 with KV Cache on/off.
2. Compute KV Cache memory for a 32K context in LLaMA-7B.
3. Compare KV Cache memory savings from GQA: LLaMA-7B (32 heads) vs LLaMA-70B (8 KV heads).
4. Simulate throughput improvement from continuous batching.
5. Explain how PagedAttention solves memory fragmentation.

> Solutions: `solutions/ch23_solutions.ipynb`
